# Projet Data Scientist: Prédiction de la Qualité de l'Air

## Objectif
Prédire la qualité de l'air dans une ville intelligente en utilisant des données météorologiques
et des caractéristiques des localisations.

## Dataset
- **Train.csv**: Données d'entraînement avec variables météorologiques et qualité de l'air
- **Test.csv**: Données de test pour les prédictions
- **airqo_metadata.csv**: Métadonnées des localisations

## Approche
1. Feature engineering sur les séries temporelles météorologiques
2. Intégration des métadonnées des localisations
3. Entraînement de modèles avancés (LightGBM, XGBoost, CatBoost)
4. Dashboard Streamlit pour le déploiement

In [16]:
# Import des bibliothèques nécessaires
import joblib
import joblib
import math
import warnings

import plotly.express as px


import catboost as cat
import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold

warnings.filterwarnings("ignore")

In [2]:
# Chargement des données
print("Chargement des données...")

train = pd.read_csv("Train.csv")
test = pd.read_csv("Test.csv")
sample_sub = pd.read_csv("sample_sub.csv")
metadata = pd.read_csv("airqo_metadata.csv")

print(f"Train: {train.shape}")
print(f"Test: {test.shape}")
print(f"Metadata: {metadata.shape}")

Chargement des données...
Train: (15539, 9)
Test: (5035, 8)
Metadata: (5, 17)


In [3]:
# Analyse exploratoire des données
print("Analyse exploratoire des données")

# Statistiques de la variable cible
print("\nStatistiques de la qualité de l'air (target):")
target_stats = train["target"].describe()
print(target_stats)

# Distribution par localisation
print("\nRépartition des localisations:")
location_counts = train["location"].value_counts()
print(location_counts)

# Métadonnées des localisations
print("\nMétadonnées des localisations:")
print(metadata)

Analyse exploratoire des données

Statistiques de la qualité de l'air (target):
count    15539.000000
mean        58.242429
std         42.373700
min          1.452619
25%         33.482625
50%         46.504048
75%         68.569062
max        475.820000
Name: target, dtype: float64

Répartition des localisations:
location
A    5122
D    4990
E    2907
C    1753
B     767
Name: count, dtype: int64

Métadonnées des localisations:
   Unnamed: 0 location  loc_altitude  km2  aspect  dist_motorway  dist_trunk  \
0           0        A        1122.4  1.9   194.0            NaN         NaN   
1           1        B        1155.4  5.4   219.8            NaN  528.078476   
2           2        C        1178.3  8.5   168.7            NaN   32.885520   
3           3        D         980.8  0.8    90.0            NaN         NaN   
4           4        E        1186.5  1.6   121.0            NaN  850.423131   

   dist_primary  dist_secondary  dist_tertiary  dist_unclassified  \
0     14.695789 

In [6]:
# Conversion des features météorologiques
def replace_nan(x):
    if x == " ":
        return np.nan
    else:
        return float(x)

features = ["temp", "precip", "rel_humidity", "wind_dir", "wind_spd", "atmos_press"]

print("Conversion des features météorologiques...")


def extract_features(df, features):
    for feature in features:
        df[f"{feature}_mean"] = df[feature].apply(lambda x: np.nanmean(x))
        df[f"{feature}_std"] = df[feature].apply(lambda x: np.nanstd(x))
        df[f"{feature}_min"] = df[feature].apply(lambda x: np.nanmin(x))
        df[f"{feature}_max"] = df[feature].apply(lambda x: np.nanmax(x))
    return df

train = extract_features(train, features)
test = extract_features(test, features)

# Préparation finale des données
def prepare_final_data(train, test, features):
    """Prépare les données finales pour l'entraînement"""

    # Suppression des colonnes de listes originales
    data = pd.concat([train, test], sort=False).reset_index(drop=True)
    data.drop(features, axis=1, inplace=True)

    # Séparation train/test
    train_final = data[data["target"].notnull()].reset_index(drop=True)
    test_final = data[data["target"].isna()].reset_index(drop=True)

    # Suppression des colonnes inutiles
    cols_to_drop = ["ID", "location"]
    train_final = train_final.drop(
        columns=[col for col in cols_to_drop if col in train_final.columns]
    )
    test_final = test_final.drop(
        columns=[col for col in cols_to_drop if col in test_final.columns]
    )

    return train_final, test_final

print("Préparation finale des données...")
train_final, test_final = prepare_final_data(train, test, features)

print(f"Données finales - Train: {train_final.shape}, Test: {test_final.shape}")
print(f"Features disponibles: {len(train_final.columns)}")

Conversion des features météorologiques...
Préparation finale des données...
Données finales - Train: (15539, 25), Test: (5035, 25)
Features disponibles: 25


In [7]:
# Configuration de la validation croisée
print("Configuration de la validation croisée...")

target_name = "target"
n_splits = 5

# Préparation des features
feature_cols = [col for col in train_final.columns if col != target_name]

# Configuration de la validation croisée
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
train_final["fold"] = 0

for fold, (train_idx, val_idx) in enumerate(kf.split(train_final)):
    train_final.loc[val_idx, "fold"] = fold

print(f"Validation croisée configurée avec {n_splits} folds")
print(f"Nombre de features: {len(feature_cols)}")

Configuration de la validation croisée...
Validation croisée configurée avec 5 folds
Nombre de features: 24


In [8]:
# Métriques d'évaluation
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)

def r2(y_true, y_pred):
    return r2_score(y_true, y_pred)

def evaluate_model(y_true, y_pred):
    """Évalue un modèle avec plusieurs métriques"""
    return {
        "RMSE": rmse(y_true, y_pred),
        "MAE": mae(y_true, y_pred),
        "R2": r2(y_true, y_pred),
    }

print("Métriques d'évaluation définies")

Métriques d'évaluation définies


In [9]:
# Entraînement du modèle LightGBM
def train_lgbm_model(train_data, features, target_name):
    """Entraîne un modèle LightGBM avec validation croisée"""

    oof_predictions = np.zeros(len(train_data))
    scores = []

    for fold in range(5):
        print(f"Fold {fold + 1}/5")

        # Données d'entraînement et de validation
        train_fold = train_data[train_data["fold"] != fold]
        val_fold = train_data[train_data["fold"] == fold]

        X_train = train_fold[features]
        y_train = train_fold[target_name]
        X_val = val_fold[features]
        y_val = val_fold[target_name]

        # Paramètres LightGBM
        params = {
            "objective": "regression",
            "metric": "rmse",
            "boosting_type": "gbdt",
            "learning_rate": 0.01,
            "max_depth": 8,
            "num_leaves": 31,
            "feature_fraction": 0.8,
            "bagging_fraction": 0.8,
            "bagging_freq": 2,
            "random_state": 42,
            "verbosity": -1,
        }

        # Création des datasets
        train_dataset = lgb.Dataset(X_train, label=y_train)
        val_dataset = lgb.Dataset(X_val, label=y_val, reference=train_dataset)

        # Entraînement
        model = lgb.train(
            params=params,
            train_set=train_dataset,
            valid_sets=[train_dataset, val_dataset],
            num_boost_round=10000,
            callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)],
        )

        # Prédictions
        val_pred = model.predict(X_val, num_iteration=model.best_iteration)
        oof_predictions[val_fold.index] = val_pred

        # Score
        fold_score = rmse(y_val, val_pred)
        scores.append(fold_score)
        print(f"Fold {fold + 1} RMSE: {fold_score:.4f}")

    overall_score = rmse(train_data[target_name], oof_predictions)
    print(f"Score global RMSE: {overall_score:.4f}")

    return oof_predictions, scores, overall_score

print("Entraînement du modèle LightGBM...")
lgbm_oof, lgbm_scores, lgbm_rmse = train_lgbm_model(
    train_final, feature_cols, target_name
)

Entraînement du modèle LightGBM...
Fold 1/5
Training until validation scores don't improve for 100 rounds
[200]	training's rmse: 32.9643	valid_1's rmse: 35.737
[400]	training's rmse: 30.5469	valid_1's rmse: 33.8636
[600]	training's rmse: 29.1328	valid_1's rmse: 32.8959
[800]	training's rmse: 28.0718	valid_1's rmse: 32.1678
[1000]	training's rmse: 27.1808	valid_1's rmse: 31.6231
[1200]	training's rmse: 26.3983	valid_1's rmse: 31.1731
[1400]	training's rmse: 25.713	valid_1's rmse: 30.7602
[1600]	training's rmse: 25.0979	valid_1's rmse: 30.4121
[1800]	training's rmse: 24.52	valid_1's rmse: 30.1406
[2000]	training's rmse: 24.0017	valid_1's rmse: 29.8772
[2200]	training's rmse: 23.5323	valid_1's rmse: 29.6591
[2400]	training's rmse: 23.1019	valid_1's rmse: 29.45
[2600]	training's rmse: 22.6874	valid_1's rmse: 29.2474
[2800]	training's rmse: 22.2974	valid_1's rmse: 29.0783
[3000]	training's rmse: 21.931	valid_1's rmse: 28.9376
[3200]	training's rmse: 21.589	valid_1's rmse: 28.79
[3400]	train

In [10]:
# Entraînement du modèle XGBoost
def train_xgb_model(train_data, features, target_name):
    """Entraîne un modèle XGBoost avec validation croisée"""

    oof_predictions = np.zeros(len(train_data))
    scores = []

    for fold in range(5):
        print(f"Fold {fold + 1}/5")

        train_fold = train_data[train_data["fold"] != fold]
        val_fold = train_data[train_data["fold"] == fold]

        X_train = train_fold[features]
        y_train = train_fold[target_name]
        X_val = val_fold[features]
        y_val = val_fold[target_name]

        # Paramètres XGBoost
        params = {
            "objective": "reg:squarederror",
            "eval_metric": "rmse",
            "learning_rate": 0.01,
            "max_depth": 8,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "random_state": 42,
            "verbosity": 0,
        }

        # Entraînement
        model = xgb.train(
            params=params,
            dtrain=xgb.DMatrix(X_train, label=y_train),
            num_boost_round=10000,
            evals=[(xgb.DMatrix(X_val, label=y_val), "validation")],
            early_stopping_rounds=100,
            verbose_eval=200,
        )

        # Prédictions
        val_pred = model.predict(
            xgb.DMatrix(X_val), iteration_range=(0, model.best_iteration)
        )
        oof_predictions[val_fold.index] = val_pred

        fold_score = rmse(y_val, val_pred)
        scores.append(fold_score)
        print(f"Fold {fold + 1} RMSE: {fold_score:.4f}")

    overall_score = rmse(train_data[target_name], oof_predictions)
    print(f"Score global RMSE: {overall_score:.4f}")

    return oof_predictions, scores, overall_score

print("Entraînement du modèle XGBoost...")
xgb_oof, xgb_scores, xgb_rmse = train_xgb_model(
    train_final, feature_cols, target_name
)

Entraînement du modèle XGBoost...
Fold 1/5
[0]	validation-rmse:43.92836
[200]	validation-rmse:32.71844
[400]	validation-rmse:30.33956
[600]	validation-rmse:29.37481
[800]	validation-rmse:28.74026
[1000]	validation-rmse:28.31783
[1200]	validation-rmse:27.96230
[1400]	validation-rmse:27.68701
[1600]	validation-rmse:27.46206
[1800]	validation-rmse:27.28975
[2000]	validation-rmse:27.13037
[2200]	validation-rmse:26.98759
[2400]	validation-rmse:26.87331
[2600]	validation-rmse:26.77813
[2800]	validation-rmse:26.69693
[3000]	validation-rmse:26.62007
[3200]	validation-rmse:26.55344
[3400]	validation-rmse:26.49993
[3600]	validation-rmse:26.43845
[3800]	validation-rmse:26.40322
[4000]	validation-rmse:26.35439
[4200]	validation-rmse:26.31105
[4400]	validation-rmse:26.27782
[4600]	validation-rmse:26.23701
[4800]	validation-rmse:26.20250
[5000]	validation-rmse:26.18008
[5200]	validation-rmse:26.15426
[5400]	validation-rmse:26.14175
[5600]	validation-rmse:26.12221
[5800]	validation-rmse:26.10071
[600

In [11]:
# Entraînement du modèle CatBoost
def train_catboost_model(train_data, features, target_name):
    """Entraîne un modèle CatBoost avec validation croisée"""

    oof_predictions = np.zeros(len(train_data))
    scores = []

    for fold in range(5):
        print(f"Fold {fold + 1}/5")

        train_fold = train_data[train_data["fold"] != fold]
        val_fold = train_data[train_data["fold"] == fold]

        X_train = train_fold[features]
        y_train = train_fold[target_name]
        X_val = val_fold[features]
        y_val = val_fold[target_name]

        # Paramètres CatBoost
        model = cat.CatBoostRegressor(
            iterations=10000,
            learning_rate=0.01,
            depth=8,
            random_state=42,
            verbose=200,
            early_stopping_rounds=100,
        )

        # Entraînement
        model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)

        # Prédictions
        val_pred = model.predict(X_val)
        oof_predictions[val_fold.index] = val_pred

        fold_score = rmse(y_val, val_pred)
        scores.append(fold_score)
        print(f"Fold {fold + 1} RMSE: {fold_score:.4f}")

    overall_score = rmse(train_data[target_name], oof_predictions)
    print(f"Score global RMSE: {overall_score:.4f}")

    return oof_predictions, scores, overall_score

print("Entraînement du modèle CatBoost...")
cat_oof, cat_scores, cat_rmse = train_catboost_model(
    train_final, feature_cols, target_name
)

Entraînement du modèle CatBoost...
Fold 1/5
0:	learn: 41.8501296	test: 43.9653722	best: 43.9653722 (0)	total: 65.1ms	remaining: 10m 51s
200:	learn: 35.0962255	test: 37.3931291	best: 37.3931291 (200)	total: 1.13s	remaining: 55s
400:	learn: 33.1798608	test: 35.9286506	best: 35.9286506 (400)	total: 2.1s	remaining: 50.2s
600:	learn: 31.9774085	test: 35.0380022	best: 35.0380022 (600)	total: 3.09s	remaining: 48.4s
800:	learn: 31.0376869	test: 34.3894991	best: 34.3894991 (800)	total: 4.04s	remaining: 46.3s
1000:	learn: 30.2002453	test: 33.8200551	best: 33.8200551 (1000)	total: 4.89s	remaining: 44s
1200:	learn: 29.4995351	test: 33.3770031	best: 33.3770031 (1200)	total: 5.78s	remaining: 42.3s
1400:	learn: 28.8492348	test: 32.9693009	best: 32.9693009 (1400)	total: 6.73s	remaining: 41.3s
1600:	learn: 28.2830995	test: 32.6248734	best: 32.6248734 (1600)	total: 7.64s	remaining: 40.1s
1800:	learn: 27.7303290	test: 32.3184486	best: 32.3184486 (1800)	total: 8.59s	remaining: 39.1s
2000:	learn: 27.227444

In [12]:
# Comparaison des modèles
print("Comparaison des modèles:")
print(f"LightGBM RMSE: {lgbm_rmse:.4f}")
print(f"XGBoost RMSE: {xgb_rmse:.4f}")
print(f"CatBoost RMSE: {cat_rmse:.4f}")

# Meilleur modèle
models_scores = {"LightGBM": lgbm_rmse, "XGBoost": xgb_rmse, "CatBoost": cat_rmse}

best_model = min(models_scores, key=models_scores.get)
print(f"\nMeilleur modèle: {best_model} (RMSE: {models_scores[best_model]:.4f})")

Comparaison des modèles:
LightGBM RMSE: 28.0238
XGBoost RMSE: 26.9405
CatBoost RMSE: 28.6719

Meilleur modèle: XGBoost (RMSE: 26.9405)


In [13]:
# Entraînement du modèle final sur toutes les données
def train_final_model(train_data, test_data, features, target_name, model_type):
    """Entraîne le modèle final sur toutes les données"""

    X_train = train_data[features]
    y_train = train_data[target_name]
    X_test = test_data[features]

    print(f"Entraînement du modèle final {model_type}...")

    if model_type == "LightGBM":
        params = {
            "objective": "regression",
            "metric": "rmse",
            "boosting_type": "gbdt",
            "learning_rate": 0.01,
            "max_depth": 8,
            "num_leaves": 31,
            "feature_fraction": 0.8,
            "bagging_fraction": 0.8,
            "bagging_freq": 2,
            "random_state": 42,
            "verbosity": -1,
        }

        train_dataset = lgb.Dataset(X_train, label=y_train)
        model = lgb.train(
            params=params, train_set=train_dataset, num_boost_round=5000
        )

        test_pred = model.predict(X_test)

    elif model_type == "XGBoost":
        params = {
            "objective": "reg:squarederror",
            "eval_metric": "rmse",
            "learning_rate": 0.01,
            "max_depth": 8,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "random_state": 42,
            "verbosity": 0,
        }

        model = xgb.train(
            params=params,
            dtrain=xgb.DMatrix(X_train, label=y_train),
            num_boost_round=5000,
        )

        test_pred = model.predict(xgb.DMatrix(X_test))

    else:  # CatBoost
        model = cat.CatBoostRegressor(
            iterations=5000, learning_rate=0.01, depth=8, random_state=42, verbose=0
        )

        model.fit(X_train, y_train)
        test_pred = model.predict(X_test)

    return test_pred, model

# Entraînement du meilleur modèle
final_predictions, final_model = train_final_model(
    train_final, test_final, feature_cols, target_name, best_model
)

print(f"Modèle final {best_model} entraîné")

Entraînement du modèle final XGBoost...
Modèle final XGBoost entraîné


In [15]:
joblib.dump({"model": final_model, "features": feature_cols}, "pipeline.pkl")
print("Pipeline sauvegardé dans pipeline.pkl")

Pipeline sauvegardé dans pipeline.pkl


# Analyse d'importance des features avec SHAP

In [18]:
# Importance des features
importance = final_model.get_score(importance_type="gain")


# Convertir en DataFrame
importance_df = pd.DataFrame(
    {"feature": list(importance.keys()), "importance": list(importance.values())}
)

# Trier
importance_df = importance_df.sort_values(by="importance", ascending=True)

fig = px.bar(
    importance_df.head(20),
    x="importance",
    y="feature",
    orientation="h",
    title="Top 20 Feature Importance",
)

fig.show()

On peut voir ici le top des features qui impactent nos prédictions. Ce sont généralement des features météo. La pression atmosphérique(max) a un impact significatif par rapport aux autres features.

# Résultats et Prochaines Étapes

## Performance des Modèles
Les trois modèles ont été entraînés avec validation croisée 5-fold:
- **LightGBM**: Excellent pour les données tabulaires avec features numériques
- **XGBoost**: Robuste et performant sur les données structurées
- **CatBoost**: Gère bien les features catégorielles et les valeurs manquantes

Modèle gagnant: Xgboost avec RMSE = 26.94


## Améliorations Possibles
- Feature engineering plus avancé (moyennes glissantes, détection de tendances)
- Hyperparameter tuning avec Optuna
- Ensembling des modèles
